In [1]:
import json 


In [2]:
from collections import defaultdict


def obtain_incorrect_prompts(data_path):
    data = [json.loads(line) for line in open(data_path)]
    prompts2items = defaultdict(list)
    for item in data:
        prompts2items[item['input']].append(item)
    incorrect_items = {k: v for k, v in prompts2items.items() if all([item['score'] == 0 for item in v])}
    return list(incorrect_items.values())

In [3]:
incorrect_items = obtain_incorrect_prompts("/mnt/petrelfs/jiangshuyang/repo/efficient_RL/rollout_data/verl_math/deepscaler_qwen34b_grpo_neg/150.jsonl")


In [57]:
test_item = incorrect_items[3]
test_item

[{'input': 'A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. The assistant thinks deeply and output the final answer within \\boxed{}. \nUser: A triangle $H$ is inscribed in a regular hexagon $S$ such that one side of $H$ is parallel to one side of $S$. What is the maximum possible ratio of the area of $H$ to the area of $S$?\nAssistant: ',
  'output': '1. **Understanding the Problem:**\n   \n   We are given a regular hexagon \\( S \\) and a triangle \\( H \\) inscribed in \\( S \\) such that one side of \\( H \\) is parallel to one side of \\( S \\). We need to find the maximum possible ratio of the area of \\( H \\) to the area of \\( S \\).\n\n2. **Properties of a Regular Hexagon:**\n   \n   A regular hexagon can be divided into 6 equilateral triangles. Let’s denote the side length of the hexagon as \\( s \\). Therefore, the 

In [5]:
from transformers import AutoTokenizer 
from rollout.vllm_rollout import VLLMRollout


In [6]:
model = VLLMRollout("/mnt/phwfile/medai_p/LLMModels/LLMs/Qwen3-4B-Base", )


`torch_dtype` is deprecated! Use `dtype` instead!


/mnt/phwfile/medai_p/LLMModels/LLMs/Qwen3-4B-Base/tokenizer.json
INFO 10-13 14:24:58 [config.py:717] This model supports multiple tasks: {'score', 'reward', 'generate', 'embed', 'classify'}. Defaulting to 'generate'.
INFO 10-13 14:24:58 [config.py:2003] Chunked prefill is enabled with max_num_batched_tokens=32768.
INFO 10-13 14:25:00 [core.py:58] Initializing a V1 LLM engine (v0.8.5.post1) with config: model='/mnt/phwfile/medai_p/LLMModels/LLMs/Qwen3-4B-Base', speculative_config=None, tokenizer='/mnt/phwfile/medai_p/LLMModels/LLMs/Qwen3-4B-Base', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=16384, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='auto', 

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 10-13 14:25:06 [loader.py:458] Loading weights took 2.88 seconds
INFO 10-13 14:25:06 [gpu_model_runner.py:1347] Model loading took 7.5532 GiB and 3.878053 seconds
INFO 10-13 14:25:18 [backends.py:420] Using cache directory: /mnt/petrelfs/jiangshuyang/.cache/vllm/torch_compile_cache/5390603ccd/rank_0_0 for vLLM's torch.compile
INFO 10-13 14:25:18 [backends.py:430] Dynamo bytecode transform time: 12.22 s
INFO 10-13 14:25:29 [backends.py:118] Directly load the compiled graph(s) for shape None from the cache, took 9.473 s
INFO 10-13 14:25:37 [monitor.py:33] torch.compile takes 12.22 s in total
INFO 10-13 14:25:38 [kv_cache_utils.py:634] GPU KV cache size: 441,488 tokens
INFO 10-13 14:25:38 [kv_cache_utils.py:637] Maximum concurrency for 16,384 tokens per request: 26.95x
INFO 10-13 14:26:08 [gpu_model_runner.py:1686] Graph capturing finished in 30 secs, took 2.12 GiB
INFO 10-13 14:26:08 [core.py:159] init engine (profile, create kv cache, warmup model) took 61.71 seconds
INFO 10-13 14:

In [58]:
prompt = test_item[1]['input'].split("User: ")[1].split("\n\nAssistant: ")[0] + "\nPlease reason step by step and put your final answer within \\boxed{}."
prompt = test_item[0]['input']
# prompt

results = model.rollout(prompt, wrap_chat=False, n=64, top_p=0.95, temperature=1.1)


In [51]:
# model.rollout(test_item[1]['input'].split("User: ")[1].split("\n\nAssistant: ")[0])
from verl.utils.reward_score.math_verify import compute_score

In [56]:
results['response_text']
# [len(r) for r in results['response_ids']]
# import re 
# def extract_boxed(text):
#     match = re.search(r'\\boxed\{(.*?)\}', text)
#     if match:
#         return match.group(1)
#     return None

# extracted_results = [compute_score(r, test_item[0]['gts']) for r in results['response_text']]
# extracted_results
# extracted_results = [ for r in results['response_text']]
# print(results['response_text'])
# print(results['response_text'][4])

["1. **Initial Setup:** We start by identifying the relationship between the longer and shorter sides of the rectangle formed by the fence. The problem states that the longer side of the garden will have three times as many fence posts as the shorter side.\n\n2. **Representing the problem algebraically:** \n   - Let's denote the number of fence posts along the shorter side of the rectangle as \\( y \\). \n   - Consequently, the number of posts along the longer side would then be \\( 3y \\).\n\n3. **Number of posts including the corners:** Each corner would be counted twice, once for each adjacent side. Therefore, if we are counting all posts including the corners, the total number of posts is \\( 2y + 2 \\times 3y = 2y + 6y = 8y \\). However, for rectangle edges, we need to include the corners.\n\n4. **Setting the equation for the total number of posts:** Jennifer has 24 fence posts. Therefore:\n   \\[\n   8y = 24 \\implies y = \\frac{24}{8} = 3\n   \\]\n   Hence, the shorter side has 

 
   \[
   g(4) = 4(f(1) + g(1)) = 4(2 + 1) = 12, g(5) = 4(f(2) + g(2)) = 4(6 + 2) = 32, g(6) = 4(f(3) + g(3)) = 4(16 + 4) = 80, g(7) = 4(f(4) + g(4)) = 4(64 + 12) = 304, g(8) = 4(f(5) + g(5)) = 4(208 + 32) = 1000, g(9) = 4(f(6) + g(6)) = 4(800 + 80) = 3520, g(10) = 4(f(7) + g(7)) = 4(1664 + 304) = 7872
   \] 
6. The total number of $10$-letter sequences that satisfy the condition is $f(10) + g(10)$. We compute $f(10)$ using the recurrence relation:
   \[
   f(10) = 2(f(9) + g(9)) = 2(3520 + 7872) = 2 \times 11392 = 22784
   \] 
   So the total number of valid sequences is $f(10) + g(10) = 22784 + 7872 = 30656$.
7. Finally, we find the remainder when $30656$ is divided by $1000$:
   \[
   30656 \mod 1000 = 656
   \]
   
The final answer is $\boxed{656}$.
 
6. Using these relations, we compute the values for $n=10$:
   \[
   f(4)=46, g(4)=18, f(5)=136, g(5)=56, f(6)=376, g(6)=168, f(7)=1040, g(7)=464, f(8)=2912, g(8)=1232
   \]
   \[
   f(9)=8224, g(9)=3568, f(10)=22816, g(10)=10240
   

In [1]:
import json 
from math_verify import parse 


In [2]:
data_path = "/mnt/petrelfs/jiangshuyang/repo/efficient_RL/rollout_data/verl_math/deepscaler_qwen34b_grpo_dynamicrollout/53.jsonl"
data = [json.loads(line) for line in open(data_path)]


In [9]:
question_groups = {}
for item in data:
    if item['input'] not in question_groups:
        question_groups[item['input']] = []
    question_groups[item['input']].append(item)

# count how many questions that have 64 samples, with at least one correct answer
oversample_correct_problems = {k: v[:16] for k, v in question_groups.items() if len(v) >= 16 and any([item['score'] == 1 for item in v])}
oversample_incorrect_problems = {k: v[:16] for k, v in question_groups.items() if len(v) >= 16 and all([item['score'] == 0 for item in v])}

In [10]:
len(oversample_correct_problems)

6

In [11]:
len(oversample_incorrect_problems)

9

In [ ]:
# for each oversample_correct_problem, count the answer variance
# for each oversample_incorrect_problem, count the answer variance
# use parse to extract the answer 
from collections import Counter


def string_variance(strings):
    """基于频率的字符串方差"""
    if not strings:
        return 0
    
    n = len(strings)
    freq = Counter(strings)
    
    # 计算 Σ(f_j/n)²
    sum_sq_freq = sum((count/n)**2 for count in freq.values())
    
    return 1 - sum_sq_freq

correct_variance = []
incorrect_variance = []
for k, v in oversample_correct_problems.items():
    answers = [str(parse(item['output'])[0]) if parse(item['output']) is not None else "None" for item in v ]
    # print(answers)
    if len(answers) > 1:
        correct_variance.append(string_variance(answers))
for k, v in oversample_incorrect_problems.items():
    answers = [str(parse(item['output'])[0]) if parse(item['output']) is not None else "None" for item in v ]
    if len(answers) > 1:
        incorrect_variance.append(string_variance(answers))

        

['2', 'h', '9/8', '(3 + 2*sqrt(3))/2', '(3*sqrt(2))/8', '1 + sqrt(3)', '(-1*2*sqrt(3) + 5)/2', '(3 + 2*sqrt(3))/4', 'sqrt(6)/4', '2', '1', '(x**2 - 1)/((2*sqrt(3)))', '2', '3/4', '(1 + sqrt(2))/2', '2.19000000000000', '(-sqrt(6) + 3*sqrt(2))/2', '(3 + 2*sqrt(3))/4', '(3 + 2*sqrt(3))/4', 'sqrt(3)/2', 'sqrt(3)/2', '(1 + sqrt(2))/2', '(-sqrt(6) - 1 + sqrt(3) + 2*sqrt(2))/2', 'sqrt(-1 + sqrt(3))', '(5*sqrt(3))/6 + 13/9', '-1*2400*sqrt(2) + 1400*sqrt(6)', '(sqrt(2) + sqrt(3) + 2 + sqrt(6))/4', '1/2', 'Eq(c*d, a*b + 2*d*p) & Eq(2*2*sin(theta) + 1, 4*sin(theta) + 1) & Eq(a*b + 2*d*p, 2*2*sin(theta) + 1)', '1', '3 - sqrt(3)', '(1 + sqrt(2))/2']
['2', '2', '0', '-4', '-1', '3', '2', '-3', '-3', '4', '-3', '-2', '0', '-1', '0', '1', '1', '0', '17', '2', '(-1)**i*((factorial(n)*factorial(n))/((factorial(i)*factorial(-i + n)*factorial(i + 2)*factorial(-i + n - 2))))/((factorial(n)**2/(((i + 1)**2*(-i + n - 1)**2*factorial(i + 1)*factorial(-i + n - 1)))))', '2017', '4', '6', '-1', '-4', '0', '0', '

In [4]:
from evaluation.models.base_model import Local_Model
model = Local_Model("/mnt/petrelfs/jiangshuyang/repo/efficient_RL/checkpoints/verl_math/deepscaler_qwen34b_grpo_dynamicrollout/global_step_53/actor")


Got device mesh tensor([0, 1, 2, 3], dtype=torch.int32), mesh_dim_names ('fsdp',)
Processing model shards with 4 (4,) in total


`torch_dtype` is deprecated! Use `dtype` instead!


Writing to local disk
Saving model to /tmp/tmps_fyho8_
Files in the tmp dir: ['config.json', 'generation_config.json', 'model-00001-of-00002.safetensors', 'model-00002-of-00002.safetensors', 'model.safetensors.index.json', 'merges.txt', 'chat_template.jinja', 'tokenizer_config.json', 'special_tokens_map.json', 'added_tokens.json', 'vocab.json', 'tokenizer.json']


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
# the hidden state similarity within the group 
import torch 
tokenizer = model.tokenizer
def get_hidden_states(model, tokenizer, input, output, wrap_chat=False):
    
    inputs = tokenizer(input, add_special_tokens=False, return_tensors="pt").to(model.device)
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]
    
    outputs =  tokenizer(output, add_special_tokens=False, return_tensors="pt").to(model.device)
    output_ids = outputs["input_ids"]
    output_attention_mask = outputs["attention_mask"]
    
    # concatenate input and output
    seq_ids = torch.cat([input_ids, output_ids], dim=-1)
    seq_attention_mask = torch.cat([attention_mask, output_attention_mask], dim=-1)
    # print(seq_ids)
    with torch.no_grad():
        outputs = model.model(input_ids=seq_ids, attention_mask=seq_attention_mask, output_hidden_states=True)
    
    resposne_length = output_ids.size(1)
    
    hidden_states = outputs.hidden_states[-1].squeeze(0)  # (seq_len, hidden_size)
    response_hidden_states = hidden_states[-resposne_length:]  # (response_len, hidden_size)
    return response_hidden_states.mean(dim=0)  # ( hidden_size)

def get_batch_hidden_states(model, tokenizer, inputs, outputs):
    batch_inputs = tokenizer(inputs, add_special_tokens=False, return_tensors='pt', padding=True, padding_side='left').to(model.device)
    batch_outputs = tokenizer(outputs, add_special_tokens=False, return_tensors='pt', padding=True, padding_side='right').to(model.device)
    batch_input_ids = batch_inputs['input_ids']
    batch_attention_mask = batch_inputs['attention_mask']
    batch_output_ids = batch_outputs['input_ids']
    batch_attention_mask = batch_outputs['attention_mask']
    
    seq_ids = torch.cat([batch_input_ids, batch_output_ids], dim=-1)
    seq_attention_mask = torch.cat([batch_attention_mask, batch_attention_mask], dim=-1)
    hidden_states_list = []
    batch_size = 4
    for i in range(0, seq_ids.size(0), batch_size):
        batch_seq_ids = seq_ids[i:i+batch_size]
        batch_seq_attention_mask = seq_attention_mask[i:i+batch_size]
        with torch.no_grad():
            outputs = model.model(input_ids=batch_seq_ids, attention_mask=batch_seq_attention_mask, output_hidden_states=True)
            hidden_states = outputs.hidden_states[-1]

            hidden_states_list.append(hidden_states)
    
        
        # outputs = model.model(input_ids=seq_ids, attention_mask=seq_attention_mask, output_hidden_states=True)
        # hidden_states = outputs.hidden_states[-1]
    hidden_states = torch.cat(hidden_states_list, dim=0)  # (batch_size, seq_len, hidden_size)
    response_length = batch_output_ids.size(1)
    response_hidden_states = hidden_states[:, -response_length:, :]  # (batch_size, response_len, hidden_size)
    return response_hidden_states.mean(dim=1)  # (batch_size, hidden_size)

def get_similarity(model, tokenizer, items):
    hidden_states_list = []
    # inputs = [item['input'] for item in items]
    # outputs = [item['output'] for item in items]
    # hidden_states_list = get_batch_hidden_states(model, tokenizer, inputs, outputs)
    for item in items:
        input = item['input']
        output = item['output']
        hidden_states = get_hidden_states(model, tokenizer, input, output)
        hidden_states_list.append(hidden_states)
    
    # 计算余弦相似度矩阵
    hidden_states_list = torch.stack(hidden_states_list)
    similarity_matrix = torch.nn.functional.cosine_similarity(hidden_states_list.unsqueeze(1), hidden_states_list.unsqueeze(0), dim=-1)
    return similarity_matrix
    
def get_reasoning_variance(model, tokenizer, items):
    similarity_matrix = get_similarity(model, tokenizer, items)
    # compute the variance for the upper triangle matrix (excluding the diagonal)
    n = similarity_matrix.size(0)
    if n <= 1:
        return 0
    upper_triangle_indices = torch.triu_indices(n, n, offset=1)
    upper_triangle_values = similarity_matrix[upper_triangle_indices[0], upper_triangle_indices[1]]
    variance = torch.var(upper_triangle_values).item()
    return variance


In [1]:
torch.cuda.empty_cache()

NameError: name 'torch' is not defined

In [12]:
# print(torch.cuda.memory_summary())
correct_reasoning_variance = []
for k, items in oversample_correct_problems.items():
    if len(items) > 1:
        variance = get_reasoning_variance(model, tokenizer, items)
        correct_reasoning_variance.append(variance)
incorrect_reasoning_variance = []
for k, items in oversample_incorrect_problems.items():
    if len(items) > 1:
        variance = get_reasoning_variance(model, tokenizer, items)
        incorrect_reasoning_variance.append(variance)

In [13]:
correct_reasoning_variance

[0.0016632080078125,
 0.0034332275390625,
 6.246566772460938e-05,
 9.5367431640625e-05,
 0.000579833984375,
 0.0011138916015625]

In [14]:
incorrect_reasoning_variance

[0.01312255859375,
 0.0166015625,
 0.000354766845703125,
 0.00124359130859375,
 0.000301361083984375,
 0.0003299713134765625,
 0.000377655029296875,
 7.581710815429688e-05,
 6.079673767089844e-05]

In [2]:
import torch
import time

B, L, D = 1024, 8192, 2560
emb = torch.randn(B, L, D, device='cuda', dtype=torch.float16)
mask = torch.randint(0, 2, (B, L), device='cuda')
mask[:, 0] = 1  # ensure at least one valid token

# 方法1: gather
def method_gather(emb, mask):
    last_idx = mask.sum(-1) - 1
    idx = last_idx.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, emb.size(-1))
    return torch.gather(emb, 1, idx).squeeze(1)

# 方法2: advanced indexing
def method_index(emb, mask):
    B = emb.shape[0]
    last_idx = mask.sum(-1) - 1
    return emb[torch.arange(B, device=emb.device), last_idx]

# Warmup
_ = method_gather(emb, mask)
_ = method_index(emb, mask)

# Timing
torch.cuda.synchronize()
t0 = time.time()
for _ in range(1000):
    _ = method_gather(emb, mask)
torch.cuda.synchronize()
t1 = time.time()

torch.cuda.synchronize()
t2 = time.time()
for _ in range(1000):
    _ = method_index(emb, mask)
torch.cuda.synchronize()
t3 = time.time()

print(f"Gather: {(t1 - t0)*1000:.2f} ms")
print(f"Indexing: {(t3 - t2)*1000:.2f} ms")

Gather: 247.34 ms
Indexing: 82.36 ms
